
## Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))


In [0]:
df = spark.sql("select * from catalog_databricks_ete.silver.customers_silver")

## Removing Duplicates

In [0]:
df = df.dropDuplicates(["customer_id"])

In [0]:
display(df.limit(10))

#  Slowly Changing Dimensions (SCD) Type 1


## Creating Surrogate Key


In [0]:
df = df.withColumn("DimCustomerKey",monotonically_increasing_id() +1 )
display(df)

In [0]:
# The surrogate key is used to identify the new records
# If there are 100 old records surrogate key ends at number 100
# In the new records, it should start from 101 

##  Handling Surrogate Key for Old Data


In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey,customer_id,create_date,update_date 
                          from catalog_databricks_ete.gold.DimCstomers''')
    
else:
        df_old = spark.sql('''select 0 DimCustomerKey,0 customer_id,0 create_date, 0 update_date 
                       from catalog_databricks_ete.silver.customers_silver
                       where 1=0''')

## Renaming df_old Customer Columns


In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey","old_DimCustomerKey").withColumnRenamed("customer_id","old_customer_id").withColumnRenamed("create_date","old_create_date").withColumnRenamed("update_date","old_update_date")

In [0]:
df_old.display()

##  Joining the new and the old data


In [0]:
df_join = df.join(df_old,df["customer_id"]==df_old["old_customer_id"],"left")


In [0]:
df_join.display()

##  Separating new v/s old records


In [0]:
df_new  = df_join.filter(col("old_DimCustomerKey").isNull())

In [0]:
display(df_new)

In [0]:
df_old  = df_join.filter(col("old_DimCustomerKey").isNotNull())

## Transforming df_old 

In [0]:
# Dropping unnecessary columns

df_old = df_old.drop("old_DimCustomerKey","old_customer_id","old_update_date")

# Renaming old_create_date column to create_date
df_old =df_old.withColumnRenamed("old_create_date","create_date").withColumn("create_date",to_timestamp(col("create_date")))

# Updating the update_date with the current_timestamp 
df_old=df_old.withColumn("update_date",current_timestamp())

In [0]:
display(df_old)

In [0]:
df_new = df_new.drop("old_DimCustomerKey","old_customer_id","old_update_date","old_create_date")

# Updating the create_date & the update_date with the current_timestamp as they are processed right now at this moment 
df_new=df_new.withColumn("create_date",current_timestamp()).withColumn("update_date",current_timestamp())

## Surrogate key from 1  

In [0]:
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id() +1 )


In [0]:
df_new.limit(10).display()

## Adding Max Surrogate key   

In [0]:
# If the table does not exists,  max_surrogate_key = 0
if init_load_flag==1:
    max_surrogate_key = 0

# Else,the max_surrogate key is derived from the max of DimCustomerKey old records 

else:
    df_max_surrogate = spark.sql('''select max(DimCustomerKey) as max_surrogate_key from catalog_databricks_ete.gold.DimCustomer''')

    #  Converting df_max_surrogate to max surrogate key variable 
    max_surrogate_key = df_max_surrogate.collect()[0]['max_surrogate_key']


## Converting df_max_surrogate to max surrogate key variable   

In [0]:
# Updating the DimCustomerKey with the max_surrogate_key +1
df_new = df_new.withColumn("DimCustomerKey",col("DimCustomerKey")+lit(max_surrogate_key))

In [0]:
display(df_new)

## Union of df_old and df_new   

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
display(df_final)
